# ColorrectalAI · Cuaderno de YOLO

Este cuaderno hace **todo el trabajo de YOLO** del proyecto, en dos partes:

| Parte | Qué hace | Necesita |
|-------|----------|----------|
| **1. Prueba rápida** (≈10 min) | Descarga un YOLO ya entrenado, lo prueba y lo exporta a ONNX | Nada más |
| **2. Entrenamiento con PolypGen** | Prepara el dataset, entrena, mide métricas, exporta el modelo y guarda el registro del experimento | El dataset PolypGen en tu Google Drive |

**Cómo se usa**
1. Menú **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**.
2. Ejecuta las celdas **en orden**, una por una, con el botón ▶ (o `Shift + Enter`).
3. Cuando una celda termine, lee lo que imprimió antes de pasar a la siguiente.

> ⚠️ Resultados **experimentales** con fines académicos. No constituyen un diagnóstico médico.

## 0. Preparación
Cambia solo la celda de configuración. Para la primera prueba deja todo como está.

In [ ]:
# ================= CONFIGURACIÓN (cambia solo esto) =================
MODELO_BASE = "yolo26n.pt"   # el más ligero; sirve para probar rápido
TAMANO_IMG = 640
SEMILLA = 42                 # fija los resultados para que se puedan reproducir

# --- Solo para la Parte 2 (entrenamiento) ---
EPOCAS = 50                  # 10 solo sirve para comprobar que todo corre; para el modelo final usa 50 (con T4 tarda ~1 h)
UMBRAL_CONF = 0.25           # confianza mínima para contar una imagen como "con pólipo"
RUTA_POLYPGEN = "/content/drive/MyDrive/PolypGen2021_MultiCenterData_v2.zip"   # tu archivo en Drive

# División por CENTRO (PolypGen viene de 6 centros distintos: C1 a C6).
# Los pacientes de un centro no aparecen en otro, así ningún paciente queda
# en dos grupos a la vez (lo que exige el plan del proyecto).
CENTROS_TRAIN = ["C1", "C2", "C3", "C4"]
CENTROS_VAL = ["C5"]
CENTROS_TEST = ["C6"]
INCLUIR_NEGATIVAS = True     # imágenes sin pólipo (sin máscara) como ejemplos negativos

In [ ]:
!pip install -q -U ultralytics onnxruntime

import torch, ultralytics
print("Ultralytics:", ultralytics.__version__)
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Tarjeta:", torch.cuda.get_device_name(0))
else:
    print("⚠️ Sin GPU: la Parte 1 funciona igual, pero el entrenamiento (Parte 2) será muy lento.")

---
# PARTE 1 · Prueba rápida con un YOLO ya entrenado

Este modelo detecta objetos cotidianos (personas, autobuses…), **no pólipos**. Sirve para
comprobar que YOLO funciona y para entregarle al backend un archivo `.onnx` con el que
armar y probar la inferencia real mientras se entrena el modelo de pólipos.

- `.pt` = el modelo en formato PyTorch (así se entrena).
- `.onnx` = el mismo modelo en formato portable (así lo ejecuta tu backend con ONNX Runtime).

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display

modelo_demo = YOLO(MODELO_BASE)          # descarga el modelo la primera vez
resultado = modelo_demo("https://ultralytics.com/images/bus.jpg", imgsz=TAMANO_IMG)
resultado[0].save("resultado_demo.jpg")
display(Image("resultado_demo.jpg"))     # verás cajas con etiquetas como "person" y "bus"

In [ ]:
ruta_onnx_demo = modelo_demo.export(format="onnx", imgsz=TAMANO_IMG)
print("Archivo creado:", ruta_onnx_demo)

In [ ]:
import onnxruntime as ort

def inspeccionar_onnx(ruta):
    """Muestra qué recibe y qué devuelve el modelo. COPIA ESTA SALIDA y compártela
    para poder escribir la inferencia del backend."""
    sesion = ort.InferenceSession(ruta, providers=["CPUExecutionProvider"])
    print("ENTRADAS:")
    for e in sesion.get_inputs():
        print("  ", e.name, e.shape, e.type)
    print("SALIDAS:")
    for s in sesion.get_outputs():
        print("  ", s.name, s.shape, s.type)

inspeccionar_onnx(ruta_onnx_demo)

In [ ]:
from google.colab import files
files.download(ruta_onnx_demo)      # baja el .onnx a tu computador

---
# PARTE 2 · Entrenamiento con PolypGen

### Antes de ejecutar esta parte
1. Crea una cuenta gratuita en **Synapse** y descarga el dataset **PolypGen** (licencia CC BY 4.0: hay que citarlo, Ali et al., 2023).
2. Sube la carpeta (o el `.zip`) a tu **Google Drive** y ajusta `RUTA_POLYPGEN` en la celda de configuración.

El cuaderno **adivina** cómo está organizado el dataset: busca imágenes y sus máscaras por el nombre
(una máscara tiene "mask" en su nombre o carpeta) y detecta el centro (C1…C6) en la ruta.
Si algo no calza, copia lo que imprimen las celdas 2.2 y 2.3 y se ajusta.

## 2.1 Conectar Google Drive y ubicar el dataset

In [ ]:
from google.colab import drive
import os, zipfile

drive.mount("/content/drive")

RAIZ = RUTA_POLYPGEN
if RAIZ.lower().endswith(".zip"):
    destino = "/content/polypgen"
    if not os.path.isdir(destino):
        print("Descomprimiendo (son ~2.4 GB, puede tardar varios minutos)...")
        with zipfile.ZipFile(RAIZ) as z:
            z.extractall(destino)
    RAIZ = destino
assert os.path.isdir(RAIZ), f"No encuentro la carpeta: {RAIZ}. Revisa RUTA_POLYPGEN."
print("Dataset en:", RAIZ)

## 2.2 Explorar cómo está organizado
Esta celda solo **muestra** las carpetas. Revisa que veas las imágenes y las máscaras.

In [ ]:
def mostrar_arbol(raiz, profundidad=4, max_lineas=150):
    base = raiz.rstrip("/").count("/")
    lineas = 0
    for ruta, carpetas, archivos in os.walk(raiz):
        carpetas.sort()
        nivel = ruta.count("/") - base
        if nivel >= profundidad:
            carpetas[:] = []          # no bajar más
        print("  " * nivel + os.path.basename(ruta) + f"/  ({len(archivos)} archivos)")
        lineas += 1
        if lineas >= max_lineas:
            print("  ... (recortado)")
            break

mostrar_arbol(RAIZ)

## 2.3 Convertir PolypGen al formato que entiende YOLO
YOLO necesita, por cada imagen, un `.txt` con una línea por pólipo: `0 x_centro y_centro ancho alto`
(valores entre 0 y 1). Una imagen **sin pólipo** lleva un `.txt` vacío.
Aquí se convierten las máscaras en cajas, se divide por centro y se copian los archivos.

Reglas que aplica (las verás impresas):
- **Positiva** = imagen con máscara.
- **Negativa** = imagen sin máscara dentro de una carpeta cuyo nombre contiene "neg" (negative…).
- **Excluida** = imagen sin máscara y sin ser negativa. No se usa, porque no sabemos si tiene pólipo
  (así nunca se enseña al modelo que una imagen con pólipo es "sana").

In [ ]:
import os, re, random, shutil
import cv2
import numpy as np

EXT_IMG = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
PATRON_CENTRO = re.compile(r"(?<![A-Za-z0-9])c([1-6])(?![0-9])", re.IGNORECASE)


def es_mascara(ruta_relativa):
    """Una máscara tiene 'mask' en su carpeta o en su nombre."""
    return "mask" in ruta_relativa.lower()


def es_negativa(ruta_relativa):
    """Carpetas o nombres con 'neg' (negative, negativeOnly...) = imágenes SIN pólipo."""
    return "neg" in ruta_relativa.lower()


def clave(nombre_archivo):
    """Nombre sin extensión y sin 'mask', para emparejar imagen y máscara."""
    nombre = os.path.splitext(os.path.basename(nombre_archivo))[0].lower()
    return re.sub(r"[_\-]?masks?", "", nombre)


def centro_de(ruta, raiz):
    """Detecta C1..C6 en la ruta. Si no encuentra, devuelve SIN_CENTRO."""
    rel = os.path.relpath(ruta, raiz)
    m = PATRON_CENTRO.search(rel)
    return f"C{m.group(1)}" if m else "SIN_CENTRO"


def asignar_division(centro):
    if centro in CENTROS_TRAIN:
        return "train"
    if centro in CENTROS_VAL:
        return "val"
    if centro in CENTROS_TEST:
        return "test"
    return "train"    # centros no reconocidos van a entrenamiento (se avisa más abajo)


def carpeta_normalizada(ruta, raiz):
    """Carpeta de un archivo sin las palabras 'images' / 'masks'.
    Así 'data_C1/images_C1' y 'data_C1/masks_C1' cuentan como la MISMA carpeta."""
    rel = os.path.relpath(os.path.dirname(ruta), raiz).lower()
    partes = [re.sub(r"images?|imgs?|masks?", "", p) for p in rel.split(os.sep)]
    return "/".join(p for p in partes if p.strip("_-. "))


def emparejar(raiz):
    """Devuelve (pares imagen-máscara, imágenes negativas, imágenes descartadas).
    - Imagen y máscara se emparejan si tienen el mismo nombre Y carpetas equivalentes.
    - Sin máscara + carpeta 'neg...' = negativa (sin pólipo).
    - Sin máscara y sin ser negativa = descartada (no sabemos si tiene pólipo)."""
    imagenes, mascaras = {}, {}
    for ruta, carpetas, archivos in os.walk(raiz):
        carpetas[:] = [d for d in carpetas if d != "__MACOSX"]
        for f in archivos:
            if f.startswith(".") or not f.lower().endswith(EXT_IMG):
                continue
            completa = os.path.join(ruta, f)
            destino = mascaras if es_mascara(os.path.relpath(completa, raiz)) else imagenes
            destino.setdefault((clave(f), carpeta_normalizada(completa, raiz)), []).append(completa)

    def clasificar(buscar):
        pares, negativas, descartadas = [], [], []
        for k, rutas_img in imagenes.items():
            for img in rutas_img:
                masc = buscar(k)
                if masc:
                    pares.append((img, masc))
                elif es_negativa(os.path.relpath(img, raiz)):
                    negativas.append(img)
                else:
                    descartadas.append(img)
        return pares, negativas, descartadas

    pares, negativas, descartadas = clasificar(lambda k: mascaras[k][0] if k in mascaras else None)
    if not pares:   # estructura inusual: último recurso, emparejar solo por nombre
        print("⚠️ No hubo pares con carpetas equivalentes; se empareja solo por nombre. Revisa los resultados.")
        por_nombre = {}
        for (nombre, _), rutas in mascaras.items():
            por_nombre.setdefault(nombre, []).extend(rutas)
        pares, negativas, descartadas = clasificar(
            lambda k: por_nombre[k[0]][0] if k[0] in por_nombre else None)
    return pares, negativas, descartadas


def mascara_a_lineas(ruta_mascara, area_minima=0.0005):
    """Convierte una máscara en líneas de etiqueta YOLO (una por pólipo)."""
    m = cv2.imread(ruta_mascara, cv2.IMREAD_GRAYSCALE)
    if m is None:
        raise ValueError(f"No pude leer la máscara: {ruta_mascara}")
    alto, ancho = m.shape
    _, binaria = cv2.threshold(m, 127, 255, cv2.THRESH_BINARY)
    contornos, _ = cv2.findContours(binaria, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    lineas = []
    for c in contornos:
        x, y, bw, bh = cv2.boundingRect(c)
        if (bw * bh) / (ancho * alto) < area_minima:     # ignora manchas diminutas
            continue
        lineas.append(f"0 {(x + bw / 2) / ancho:.6f} {(y + bh / 2) / alto:.6f} "
                      f"{bw / ancho:.6f} {bh / alto:.6f}")
    return lineas


def construir_dataset(raiz, salida="/content/datos"):
    pares, negativas, descartadas = emparejar(raiz)
    print(f"Imágenes con máscara (positivas): {len(pares)}")
    print(f"Imágenes negativas (carpetas 'neg...'): {len(negativas)} -> "
          f"{'se usan' if INCLUIR_NEGATIVAS else 'se ignoran'}")
    print(f"Imágenes sin máscara y sin ser negativas: {len(descartadas)} -> se EXCLUYEN "
          "(no sabemos si tienen pólipo)")
    if descartadas:
        print("   Ejemplos (cópialos si necesitas ayuda):")
        for d in descartadas[:3]:
            print("     ", os.path.relpath(d, raiz))
    if not pares:
        raise RuntimeError("No encontré pares imagen/máscara. Revisa RUTA_POLYPGEN y el árbol de la celda 2.2.")

    if os.path.isdir(salida):
        shutil.rmtree(salida)
    for d in ("train", "val", "test"):
        os.makedirs(f"{salida}/images/{d}")
        os.makedirs(f"{salida}/labels/{d}")

    trabajo = list(pares) + ([(img, None) for img in negativas] if INCLUIR_NEGATIVAS else [])
    conteo, ilegibles = {}, []
    for i, (img, masc) in enumerate(trabajo):
        try:
            lineas = mascara_a_lineas(masc) if masc else []
        except ValueError:
            ilegibles.append(masc)
            continue
        centro = centro_de(img, raiz)
        division = asignar_division(centro)
        nombre = f"{i:05d}_{os.path.splitext(os.path.basename(img))[0]}"
        ext = os.path.splitext(img)[1].lower()
        shutil.copy(img, f"{salida}/images/{division}/{nombre}{ext}")
        with open(f"{salida}/labels/{division}/{nombre}.txt", "w") as f:
            f.write("\n".join(lineas))
        pos, tot = conteo.get((division, centro), (0, 0))
        conteo[(division, centro)] = (pos + (1 if lineas else 0), tot + 1)

    print("\nResumen por grupo y centro  (con pólipo / total):")
    totales = {"train": 0, "val": 0, "test": 0}
    for (division, centro), (pos, tot) in sorted(conteo.items()):
        print(f"  {division:5s}  {centro:11s}  {pos:5d} / {tot:5d}")
        totales[division] += tot
    if ilegibles:
        print(f"\n⚠️ {len(ilegibles)} máscaras no se pudieron leer y se omitieron.")
    if any(c == "SIN_CENTRO" for (_, c) in conteo):
        print("\n⚠️ Hay imágenes sin centro detectado (SIN_CENTRO): se enviaron a train.")
    for d in ("val", "test"):
        if totales[d] == 0:
            raise RuntimeError(f"El grupo '{d}' quedó vacío. Cambia CENTROS_VAL / CENTROS_TEST en la configuración.")

    ruta_yaml = f"{salida}/dataset.yaml"
    with open(ruta_yaml, "w") as f:
        f.write(f"path: {salida}\ntrain: images/train\nval: images/val\ntest: images/test\n"
                "names:\n  0: polipo\n")
    print("\nListo:", ruta_yaml)
    return ruta_yaml

In [ ]:
RUTA_YAML = construir_dataset(RAIZ)

## 2.4 Comprobación visual (¡no te la saltes!)
Muestra imágenes de entrenamiento con la caja verde que salió de la máscara.
Si las cajas **no** rodean un pólipo, hay un error en la conversión: avisa antes de entrenar.

In [ ]:
import os, random
import cv2
import numpy as np
import matplotlib.pyplot as plt     # (se importa aquí por si Colab se reinició)


def ver_ejemplos(salida="/content/datos", division="train", n=4):
    carpeta = f"{salida}/images/{division}"
    if not os.path.isdir(carpeta):
        print("⚠️ Todavía no existe el dataset. Ejecuta antes la celda 2.3 (construir_dataset).")
        return

    def etiqueta_de(f):
        return f"{salida}/labels/{division}/{os.path.splitext(f)[0]}.txt"

    con_polipo = [f for f in os.listdir(carpeta) if os.path.getsize(etiqueta_de(f)) > 0]
    if not con_polipo:
        print("No hay imágenes con pólipo en este grupo.")
        return
    random.seed(globals().get("SEMILLA", 42))
    elegidos = random.sample(con_polipo, min(n, len(con_polipo)))

    imagenes = []
    for f in elegidos:
        bgr = cv2.imread(f"{carpeta}/{f}")
        if bgr is None:
            continue                       # imagen que no se pudo leer: se salta
        img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        for linea in open(etiqueta_de(f)).read().splitlines():
            if not linea.strip():
                continue
            _, cx, cy, bw, bh = map(float, linea.split())
            cv2.rectangle(img, (int((cx - bw / 2) * w), int((cy - bh / 2) * h)),
                          (int((cx + bw / 2) * w), int((cy + bh / 2) * h)), (0, 255, 0), 3)
        imagenes.append(img)

    if not imagenes:
        print("⚠️ No pude leer ninguna imagen de ejemplo.")
        return
    fig, ejes = plt.subplots(1, len(imagenes), figsize=(4 * len(imagenes), 4))
    for ax, img in zip(np.atleast_1d(ejes), imagenes):
        ax.imshow(img)
        ax.axis("off")
    plt.show()


ver_ejemplos()

## 2.5 Entrenar
La primera vez usa `EPOCAS = 10` solo para comprobar que todo funciona. Cuando veas que corre bien,
sube a 50 o más y vuelve a ejecutar desde esta celda.

In [ ]:
modelo = YOLO(MODELO_BASE)
modelo.train(data=RUTA_YAML, epochs=EPOCAS, imgsz=TAMANO_IMG, seed=SEMILLA,
             project="/content/runs", name="baseline", exist_ok=True, patience=15)

carpeta_entreno = str(modelo.trainer.save_dir)
ruta_best = os.path.join(carpeta_entreno, "weights", "best.pt")
print("Mejor modelo:", ruta_best)

## 2.6 Evaluar en el grupo de prueba (test)
Dos tipos de métricas, como pide el plan:
- **Detección** (mAP, precisión, recall): qué tan bien dibuja las cajas.
- **Por imagen** (sensibilidad, especificidad, F1, ROC-AUC, PR-AUC, Brier, ECE): ¿la imagen tiene pólipo o no?
  La "confianza" de la imagen es la de su caja más segura. Brier y ECE son una **estimación inicial** de calibración.

In [ ]:
mejor = YOLO(ruta_best)
m = mejor.val(data=RUTA_YAML, split="test", imgsz=TAMANO_IMG)
metricas_deteccion = {
    "precision": float(m.box.mp), "recall": float(m.box.mr),
    "mAP50": float(m.box.map50), "mAP50_95": float(m.box.map),
    "velocidad_ms_por_imagen": {k: round(float(v), 1) for k, v in m.speed.items()},
}
print(metricas_deteccion)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss


def calcular_ece(y_real, y_prob, bins=10):
    y_real, y_prob = np.asarray(y_real, float), np.asarray(y_prob, float)
    bordes = np.linspace(0, 1, bins + 1)
    ece = 0.0
    for i in range(bins):
        sel = (y_prob >= bordes[i]) & (y_prob <= bordes[i + 1]) if i == 0 \
            else (y_prob > bordes[i]) & (y_prob <= bordes[i + 1])
        if sel.any():
            ece += sel.sum() / len(y_real) * abs(y_real[sel].mean() - y_prob[sel].mean())
    return float(ece)


def metricas_por_imagen(modelo, salida="/content/datos", division="test", umbral=0.25):
    carpeta = f"{salida}/images/{division}"
    y_real, y_score = [], []
    for r in modelo.predict(source=carpeta, conf=0.001, imgsz=TAMANO_IMG, stream=True, verbose=False):
        nombre = os.path.splitext(os.path.basename(r.path))[0]
        real = os.path.getsize(f"{salida}/labels/{division}/{nombre}.txt") > 0
        score = float(r.boxes.conf.max()) if len(r.boxes) else 0.0
        y_real.append(int(real))
        y_score.append(score)

    y_real, y_score = np.array(y_real), np.array(y_score)
    y_pred = (y_score >= umbral).astype(int)
    vp = int(((y_pred == 1) & (y_real == 1)).sum())
    fp = int(((y_pred == 1) & (y_real == 0)).sum())
    vn = int(((y_pred == 0) & (y_real == 0)).sum())
    fn = int(((y_pred == 0) & (y_real == 1)).sum())
    div = lambda a, b: round(a / b, 4) if b else None
    sens, esp, prec = div(vp, vp + fn), div(vn, vn + fp), div(vp, vp + fp)
    f1 = round(2 * prec * sens / (prec + sens), 4) if prec and sens else None

    res = {"imagenes": int(len(y_real)), "con_polipo": int(y_real.sum()),
           "umbral": umbral, "VP": vp, "FP": fp, "VN": vn, "FN": fn,
           "sensibilidad": sens, "especificidad": esp, "precision": prec, "F1": f1,
           "brier": round(float(brier_score_loss(y_real, y_score)), 4),
           "ECE": round(calcular_ece(y_real, y_score), 4)}
    if len(set(y_real.tolist())) == 2:     # AUC solo existe si hay positivas y negativas
        res["ROC_AUC"] = round(float(roc_auc_score(y_real, y_score)), 4)
        res["PR_AUC"] = round(float(average_precision_score(y_real, y_score)), 4)
    else:
        res["ROC_AUC"] = res["PR_AUC"] = None
        print("⚠️ El grupo de prueba tiene un solo tipo de imagen: no se calculan ROC-AUC ni PR-AUC.")
    return res

metricas_imagen = metricas_por_imagen(mejor, umbral=UMBRAL_CONF)
for k, v in metricas_imagen.items():
    print(f"{k:14s} {v}")

## 2.7 Exportar a ONNX **y comprobar que el backend lo entiende**
Esta celda hace tres cosas:
1. Exporta `best.pt` a **ONNX** con los parámetros que necesita el backend (entrada fija de 640×640, un solo lote, sin media precisión).
2. Calcula su **SHA-256** (la suma de verificación que exige la arquitectura).
3. Lo **ejecuta con ONNX Runtime usando la misma lógica del backend** (letterbox → modelo → decodificación → NMS) y lo compara con el modelo `.pt` sobre imágenes de prueba. Si las cajas coinciden, el `.onnx` es válido para el backend.

> Las cajas no serán idénticas al 100 %: Ultralytics redondea el relleno al múltiplo de 32 y el backend rellena hasta 640×640. Se acepta una diferencia pequeña.

In [ ]:
import os, json, shutil, hashlib, glob
import numpy as np
import onnxruntime as ort
from PIL import Image

# ---------- 1) Exportar ----------
ruta_onnx = mejor.export(format="onnx", imgsz=TAMANO_IMG, simplify=True,
                         dynamic=False, half=False, batch=1)
print("ONNX creado:", ruta_onnx)


def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()


def inspeccionar_onnx(ruta):
    """Muestra qué recibe y qué devuelve el modelo."""
    sesion = ort.InferenceSession(ruta, providers=["CPUExecutionProvider"])
    print("ENTRADAS:")
    for e in sesion.get_inputs():
        print("  ", e.name, e.shape, e.type)
    print("SALIDAS:")
    for s in sesion.get_outputs():
        print("  ", s.name, s.shape, s.type)
    return sesion


sha_onnx = sha256(ruta_onnx)
print("SHA-256:", sha_onnx)
sesion = inspeccionar_onnx(ruta_onnx)


# ---------- 2) Misma lógica que el backend (ia/preprocesamiento.py y ia/posprocesamiento.py) ----------
def letterbox(img, t):
    img = img.convert("RGB"); w, h = img.size
    esc = min(t / w, t / h); nw, nh = int(round(w * esc)), int(round(h * esc))
    lienzo = Image.new("RGB", (t, t), (114, 114, 114))
    px, py = (t - nw) // 2, (t - nh) // 2
    lienzo.paste(img.resize((nw, nh), Image.BILINEAR), (px, py))
    x = np.asarray(lienzo, dtype=np.float32) / 255.0
    return np.ascontiguousarray(x.transpose(2, 0, 1)[None]), esc, px, py, w, h


def iou_uno(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1]); x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-9)


def nms(cajas, puntajes, umbral):
    orden = puntajes.argsort()[::-1]; mantener = []
    while orden.size:
        i = orden[0]; mantener.append(int(i)); resto = orden[1:]
        if resto.size == 0:
            break
        orden = resto[np.array([iou_uno(cajas[i], cajas[j]) for j in resto]) < umbral]
    return mantener


def decodificar(salida, conf_min, iou_max=0.45):
    s = salida[0] if salida.ndim == 3 else salida
    if s.shape[-1] == 6 and s.shape[0] != 6:               # formato A: (N, 6) con NMS integrado
        xyxy, conf = s[:, :4], s[:, 4]
        ok = conf >= conf_min
        return xyxy[ok], conf[ok], "A (NMS integrado)"
    if s.shape[0] < s.shape[1]:                             # formato B: (4+nc, anclas)
        s = s.T
    cxcywh, punt = s[:, :4], s[:, 4:]
    conf = punt.max(axis=1); ok = conf >= conf_min
    cxcywh, conf = cxcywh[ok], conf[ok]
    xyxy = np.stack([cxcywh[:, 0]-cxcywh[:, 2]/2, cxcywh[:, 1]-cxcywh[:, 3]/2,
                     cxcywh[:, 0]+cxcywh[:, 2]/2, cxcywh[:, 1]+cxcywh[:, 3]/2], axis=1)
    if len(conf):
        m = nms(xyxy, conf, iou_max); xyxy, conf = xyxy[m], conf[m]
    return xyxy, conf, "B (clásico, con NMS en el backend)"


def inferir_como_backend(ruta_img):
    entrada = sesion.get_inputs()[0].name
    tensor, esc, px, py, w, h = letterbox(Image.open(ruta_img), TAMANO_IMG)
    xyxy, conf, formato = decodificar(sesion.run(None, {entrada: tensor})[0], UMBRAL_CONF)
    if len(xyxy):
        xyxy = xyxy.astype(np.float32).copy()
        xyxy[:, [0, 2]] = ((xyxy[:, [0, 2]] - px) / esc).clip(0, w)
        xyxy[:, [1, 3]] = ((xyxy[:, [1, 3]] - py) / esc).clip(0, h)
    return xyxy, conf, formato


# ---------- 3) Comparar con el modelo .pt en imágenes de prueba ----------
carpeta_test = "/content/datos/images/test"
etiquetas = "/content/datos/labels/test"
con_polipo = [f for f in sorted(os.listdir(carpeta_test))
              if os.path.getsize(f"{etiquetas}/{os.path.splitext(f)[0]}.txt") > 0][:40]

mismos_conteos, ious, difs_conf, formato_salida = 0, [], [], ""
for f in con_polipo:
    ruta = f"{carpeta_test}/{f}"
    r = mejor.predict(ruta, conf=UMBRAL_CONF, imgsz=TAMANO_IMG, verbose=False)[0]
    cajas_pt = r.boxes.xyxy.cpu().numpy(); conf_pt = r.boxes.conf.cpu().numpy()
    cajas_ox, conf_ox, formato_salida = inferir_como_backend(ruta)
    mismos_conteos += int(len(cajas_pt) == len(cajas_ox))
    for cp, sp in zip(cajas_pt, conf_pt):               # cada caja del .pt busca su pareja en el ONNX
        if len(cajas_ox):
            j = int(np.argmax([iou_uno(cp, co) for co in cajas_ox]))
            ious.append(iou_uno(cp, cajas_ox[j])); difs_conf.append(abs(sp - conf_ox[j]))

n = len(con_polipo)
verificacion = {
    "imagenes_comparadas": n,
    "formato_salida": formato_salida,
    "misma_cantidad_de_cajas": f"{mismos_conteos}/{n}",
    "iou_medio_pt_vs_onnx": round(float(np.mean(ious)), 4) if ious else None,
    "diferencia_media_confianza": round(float(np.mean(difs_conf)), 4) if difs_conf else None,
}
print("\nVERIFICACIÓN ONNX vs modelo .pt (con la lógica del backend):")
for k, v in verificacion.items():
    print(f"  {k}: {v}")

ok = bool(ious) and np.mean(ious) >= 0.85 and mismos_conteos / max(n, 1) >= 0.8
print("\n✅ El ONNX es compatible con el backend." if ok else
      "\n⚠️ Hay diferencias grandes entre el ONNX y el modelo .pt. NO uses este archivo todavía: copia esta salida y se revisa.")

## 2.8 Registro del experimento y respaldo
Guarda **todo lo necesario para reproducir el experimento** (semilla, versiones, parámetros, métricas y la verificación del ONNX) y hace una copia en tu Drive por si Colab se cierra.

In [ ]:
import json, platform, datetime

registro = {
    "fecha": datetime.datetime.now().isoformat(timespec="seconds"),
    "modelo_base": MODELO_BASE, "epocas": EPOCAS, "tamano_img": TAMANO_IMG,
    "semilla": SEMILLA, "umbral_confianza": UMBRAL_CONF,
    "division_por_centro": {"train": CENTROS_TRAIN, "val": CENTROS_VAL, "test": CENTROS_TEST},
    "negativas_incluidas": INCLUIR_NEGATIVAS,
    "versiones": {"python": platform.python_version(), "torch": torch.__version__,
                  "ultralytics": ultralytics.__version__, "onnxruntime": ort.__version__},
    "metricas_deteccion": metricas_deteccion,
    "metricas_por_imagen": metricas_imagen,
    "onnx": {"archivo": "best.onnx", "sha256": sha_onnx,
             "entrada": {e.name: e.shape for e in sesion.get_inputs()},
             "salida": {s.name: s.shape for s in sesion.get_outputs()},
             "verificacion_backend": verificacion},
    "aviso": "Resultados experimentales con fines academicos. No constituyen diagnostico medico.",
}
ruta_registro = "/content/registro_experimento.json"
with open(ruta_registro, "w", encoding="utf-8") as f:
    json.dump(registro, f, indent=2, ensure_ascii=False)
print("Registro guardado:", ruta_registro)

# Respaldo en Google Drive
destino = "/content/drive/MyDrive/ColorrectalAI_modelo"
os.makedirs(destino, exist_ok=True)
for ruta in [ruta_best, ruta_onnx, ruta_registro, os.path.join(carpeta_entreno, "results.csv")]:
    if os.path.exists(ruta):
        shutil.copy(ruta, destino)
print("Respaldo en Drive:", destino)

## 2.9 Descargar el paquete para el backend
Esta celda crea **`modelo_para_backend.zip`** con dos archivos: `best.onnx` y `registro_experimento.json`, y te imprime los comandos para usarlos.

In [ ]:
import zipfile
from google.colab import files

with zipfile.ZipFile("/content/modelo_para_backend.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write(ruta_onnx, "best.onnx")
    z.write(ruta_registro, "registro_experimento.json")

print("""
================ QUÉ HACER CON EL PAQUETE ================
1) Descomprime modelo_para_backend.zip.
2) Copia best.onnx a:   backend\\modelos\\best.onnx   (reemplaza el que haya)
3) Reinicia el backend. Para que exija que sea ESTE modelo, antes de arrancar:

   $env:MODELO_SHA256="%s"
   python -m uvicorn api.main:app --reload

4) Para producción (Render): sube best.onnx a GitHub Releases y define MODELO_URL y MODELO_SHA256.
==========================================================
""" % sha_onnx)

files.download("/content/modelo_para_backend.zip")

---
## ¿Y ahora qué?
1. Copia **`best.onnx`** a `backend/modelos/` y reinicia el backend (`python -m uvicorn api.main:app --reload`).
2. Abre `http://localhost:8000/api/v1/salud`: debe decir `"modelo_listo": true`, y el campo `modelo_sha256` debe coincidir con el SHA-256 impreso arriba.
3. Prueba con imágenes de PolypGen desde la aplicación o desde `http://localhost:8000/probar`.
4. El `.onnx` final se publica en *Releases* de GitHub junto con su SHA-256.

**Cita obligatoria del dataset:** Ali, S., Jha, D., Ghatwary, N., et al. (2023). A multi-centre polyp detection and
segmentation dataset for generalisability assessment. *Scientific Data, 10*, 75. https://doi.org/10.1038/s41597-023-01981-y